In [1]:
import pandas as pd

# 1. Încărcăm fișierele de bază
results = pd.read_csv('results.csv')
races = pd.read_csv('races.csv')
drivers = pd.read_csv('drivers.csv')
qualifying = pd.read_csv('qualifying.csv')

# 2. Adăugăm fișierele noi, esențiale pentru Feature Engineering
constructors = pd.read_csv('constructors.csv')
circuits = pd.read_csv('circuits.csv')
driver_standings = pd.read_csv('driver_standings.csv')
constructor_standings = pd.read_csv('constructor_standings.csv')
status = pd.read_csv('status.csv')

# 3. Filtrăm coloanele pentru a păstra doar ce e util (să nu explodăm memoria)
results = results[['raceId', 'driverId', 'constructorId', 'grid', 'positionOrder', 'statusId']]
races = races[['raceId', 'year', 'circuitId']]
drivers = drivers[['driverId', 'driverRef']]
constructors = constructors[['constructorId', 'constructorRef']]
qualifying = qualifying[['raceId', 'driverId', 'position']] # 'position' e locul obținut în calificări
circuits = circuits[['circuitId', 'circuitRef']]
status = status[['statusId', 'status']]

# 4. Construim Dataset-ul Master (Unirea lor pas cu pas)
# Pornim de la 'results'
df = pd.merge(results, races, on='raceId', how='left')
df = pd.merge(df, circuits, on='circuitId', how='left')
df = pd.merge(df, drivers, on='driverId', how='left')
df = pd.merge(df, constructors, on='constructorId', how='left')
df = pd.merge(df, status, on='statusId', how='left')

# Atenție la calificări: unii piloți nu au participat, deci pot exista valori nule (NaN)
df = pd.merge(df, qualifying, on=['raceId', 'driverId'], how='left', suffixes=('', '_quali'))

# 5. Creăm Target-ul: a luat sau nu podium?
df['podium'] = df['positionOrder'].apply(lambda x: 1 if x in [1, 2, 3] else 0)

# Sortăm cronologic după an și ID-ul cursei (foarte important pentru ML)
df = df.sort_values(by=['year', 'raceId'])

print("Structura dataset-ului:")
print(df.info())
print("\nPrimele rânduri:")
print(df.head())

Structura dataset-ului:
<class 'pandas.DataFrame'>
Index: 26499 entries, 20024 to 26498
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   raceId          26499 non-null  int64  
 1   driverId        26499 non-null  int64  
 2   constructorId   26499 non-null  int64  
 3   grid            26499 non-null  int64  
 4   positionOrder   26499 non-null  int64  
 5   statusId        26499 non-null  int64  
 6   year            26499 non-null  int64  
 7   circuitId       26499 non-null  int64  
 8   circuitRef      26499 non-null  str    
 9   driverRef       26499 non-null  str    
 10  constructorRef  26499 non-null  str    
 11  status          26499 non-null  str    
 12  position        10234 non-null  float64
 13  podium          26499 non-null  int64  
dtypes: float64(1), int64(9), str(4)
memory usage: 3.0 MB
None

Primele rânduri:
       raceId  driverId  constructorId  grid  positionOrder  statusId  year  \

In [2]:
# Ne asigurăm că datele sunt sortate strict cronologic
df = df.sort_values(by=['year', 'raceId'])

# 1. Forma recentă a pilotului (media locurilor obținute în ultimele 3 curse)
# Folosim shift(1) pentru a lua în calcul doar cursele de dinaintea celei curente
df['driver_recent_form'] = df.groupby('driverId')['positionOrder'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)

# 2. Ritmul echipei (media locurilor obținute de constructor în ultimele 3 curse)
df['team_pace'] = df.groupby('constructorId')['positionOrder'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)

# 3. Tratarea valorilor lipsă (NaN)
# Pentru piloții sau echipele aflate la prima lor cursă, nu avem istoric. 
# Le vom asocia o valoare neutră (ex: locul 15).
df['driver_recent_form'] = df['driver_recent_form'].fillna(15)
df['team_pace'] = df['team_pace'].fillna(15)

# Pentru timpii de calificări (lipsesc acolo unde pilotul nu a participat), punem locul 20.
df['position'] = df['position'].fillna(20)

# 4. Afișăm rezultatul cu noile coloane adăugate
print(df[['year', 'driverRef', 'constructorRef', 'grid', 'driver_recent_form', 'team_pace', 'podium']].tail(10))

       year driverRef constructorRef  grid  driver_recent_form  team_pace  \
26489  2024   leclerc        ferrari     6            8.333333   4.666667   
26490  2024      ocon         alpine    10           12.333333   9.666667   
26491  2024    stroll   aston_martin    17           11.666667  11.000000   
26492  2024   tsunoda             rb    14           13.666667  14.333333   
26493  2024     albon       williams    16           14.666667  19.333333   
26494  2024    bottas         sauber    18           14.000000  14.666667   
26495  2024      zhou         sauber    20           14.666667  15.000000   
26496  2024    alonso   aston_martin    15            9.666667  13.000000   
26497  2024  sargeant       williams    19           18.333333  17.666667   
26498  2024    norris        mclaren     2            2.666667   3.666667   

       podium  
26489       0  
26490       0  
26491       0  
26492       0  
26493       0  
26494       0  
26495       0  
26496       0  
26497   

In [3]:
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report

# 1. Transformăm datele de tip text (categorice) în numere
le_driver = LabelEncoder()
le_constructor = LabelEncoder()
le_circuit = LabelEncoder()

df['driver_encoded'] = le_driver.fit_transform(df['driverRef'])
df['constructor_encoded'] = le_constructor.fit_transform(df['constructorRef'])
# Asigură-te că nu ai valori nule în circuitRef înainte de encodare
df['circuitRef'] = df['circuitRef'].astype(str)
df['circuit_encoded'] = le_circuit.fit_transform(df['circuitRef'])

# 2. Selectăm caracteristicile (Features) pe care modelul le va învăța
features = [
    'grid', 
    'driver_recent_form', 
    'team_pace', 
    'driver_encoded', 
    'constructor_encoded', 
    'circuit_encoded'
]

X = df[features]
y = df['podium']

# 3. Validare "Cross-Seasons" (Time-Series Split)
# Antrenăm modelul pe istoric (până în 2022 inclusiv)
# Îl testăm pe date complet noi pentru el (2023 - prezent)
train_mask = df['year'] <= 2022
test_mask = df['year'] > 2022

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

# 4. Inițializarea și Antrenarea modelului Gradient Boosting
print("Antrenăm modelul XGBoost...")
model = xgb.XGBClassifier(
    n_estimators=150,       # Numărul de "copaci" de decizie
    learning_rate=0.1,      # Cât de repede învață
    max_depth=5,            # Adâncimea maximă a logicii
    random_state=42
)

model.fit(X_train, y_train)

# 5. Predicții și Evaluare
predictions = model.predict(X_test)

print("\n--- REZULTATE EVALUARE (Sezoanele 2023+) ---")
print(f"Acuratețea generală a modelului: {accuracy_score(y_test, predictions):.2%}")
print("\nRaport detaliat (0 = Fără podium, 1 = Podium):")
print(classification_report(y_test, predictions))

Antrenăm modelul XGBoost...

--- REZULTATE EVALUARE (Sezoanele 2023+) ---
Acuratețea generală a modelului: 90.14%

Raport detaliat (0 = Fără podium, 1 = Podium):
              precision    recall  f1-score   support

           0       0.92      0.96      0.94       560
           1       0.73      0.55      0.62        99

    accuracy                           0.90       659
   macro avg       0.83      0.75      0.78       659
weighted avg       0.89      0.90      0.90       659



In [4]:
import numpy as np

# 1. Calculăm dezechilibrul claselor (câte non-podiumuri avem pentru fiecare podium)
# Va fi undeva în jur de 5.5 (17 mașini vs 3 mașini pe podium)
ratio = float(y_train.value_counts()[0]) / y_train.value_counts()[1]

# 2. Antrenăm un nou model, forțându-l să fie atent la Clasa 1 (scale_pos_weight)
print("Antrenăm modelul XGBoost optimizat...")
model_optim = xgb.XGBClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=5,
    scale_pos_weight=ratio,  # <--- TRUCUL 1: Penalizăm ratarea podiumurilor
    random_state=42
)
model_optim.fit(X_train, y_train)

# 3. Obținem PROBABILITĂȚILE de podium, nu decizia finală (Da/Nu)
# predict_proba returnează două coloane: [șansa de 0, șansa de 1]
# Pe noi ne interesează doar a doua coloană (șansa de a lua podium, indexul 1)
probabilitati_podium = model_optim.predict_proba(X_test)[:, 1]

# 4. TRUCUL 2: Setăm un prag mai permisiv (ex: 30%)
# Dacă modelul consideră că un pilot are cel puțin 30% șanse, îl declarăm pe podium.
prag_decizie = 0.30
predictii_permisive = (probabilitati_podium >= prag_decizie).astype(int)

# 5. Afișăm noul raport
print(f"\n--- REZULTATE CU THRESHOLD {prag_decizie*100}% ---")
print(classification_report(y_test, predictii_permisive))

Antrenăm modelul XGBoost optimizat...

--- REZULTATE CU THRESHOLD 30.0% ---
              precision    recall  f1-score   support

           0       0.99      0.71      0.82       560
           1       0.37      0.96      0.53        99

    accuracy                           0.74       659
   macro avg       0.68      0.83      0.68       659
weighted avg       0.90      0.74      0.78       659



In [5]:
import joblib

# Salvăm modelul XGBoost și Encoderele (ca să știm ce ID are fiecare pilot)
joblib.dump(model_optim, 'f1_model.pkl')
joblib.dump(le_driver, 'le_driver.pkl')
joblib.dump(le_constructor, 'le_constructor.pkl')
joblib.dump(le_circuit, 'le_circuit.pkl')

print("Model salvat cu succes!")

Model salvat cu succes!


In [6]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import precision_recall_curve
import numpy as np

print("Căutăm cei mai buni parametri pentru model... (poate dura 1-2 minute)")

# 1. Definim grila de parametri pe care modelul să îi testeze
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0],           # Previne overfitting-ul
    'colsample_bytree': [0.8, 1.0],    # Alege aleatoriu coloane
    'scale_pos_weight': [2, 3, 4]      # Testăm cât de mult să favorizăm podiumul
}

# 2. Inițializăm căutarea automată (RandomizedSearchCV)
xgb_model = xgb.XGBClassifier(random_state=42)
random_search = RandomizedSearchCV(
    estimator=xgb_model, 
    param_distributions=param_grid, 
    n_iter=20,          # Testează 20 de combinații diferite
    scoring='f1',       # Obiectivul: maximizarea Scorului F1 (Precision + Recall)
    cv=3,               # Cross-validare în 3 pași
    verbose=1,
    random_state=42,
    n_jobs=-1           # Folosește toate nucleele procesorului tău
)

# Antrenăm modelul cu testarea parametrilor
random_search.fit(X_train, y_train)

# Acesta este cel mai bun model găsit!
best_model = random_search.best_estimator_
print(f"\nCei mai buni parametri găsiți:\n{random_search.best_params_}")

# 3. Găsim PRAGUL OPTIM matematic (Optimal Threshold)
# Generăm probabilitățile pe setul de test
y_probs = best_model.predict_proba(X_test)[:, 1]

# Calculăm Precision și Recall pentru TOATE pragurile posibile (de la 0% la 100%)
precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)

# Calculăm Scorul F1 pentru fiecare prag și îl alegem pe cel mai mare
f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-10) # 1e-10 e pus ca să nu împărțim la 0
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"\nPragul optim matematic este: {optimal_threshold * 100:.1f}%")

# 4. Evaluăm modelul final cu pragul optim
final_predictions = (y_probs >= optimal_threshold).astype(int)

print(f"\n--- REZULTATE FINALE SUPREME ---")
print(classification_report(y_test, final_predictions))

# Salvăm noul model SUPERIOR pentru interfața web
import joblib
joblib.dump(best_model, 'f1_model.pkl')
print("\nModelul optimizat a fost salvat ca 'f1_model.pkl'!")

Căutăm cei mai buni parametri pentru model... (poate dura 1-2 minute)
Fitting 3 folds for each of 20 candidates, totalling 60 fits

Cei mai buni parametri găsiți:
{'subsample': 1.0, 'scale_pos_weight': 3, 'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 0.8}

Pragul optim matematic este: 60.1%

--- REZULTATE FINALE SUPREME ---
              precision    recall  f1-score   support

           0       0.96      0.92      0.94       560
           1       0.62      0.79      0.70        99

    accuracy                           0.90       659
   macro avg       0.79      0.85      0.82       659
weighted avg       0.91      0.90      0.90       659


Modelul optimizat a fost salvat ca 'f1_model.pkl'!
